<div align="center">

# Jackrong-llm-finetuning-guide 🌌
**面向初学者和开发者的教育型端到端大语言模型微调流水线**

本笔记本目前专注于在 **Kaggle 笔记本中微调 Qwen3.5-35B-A3B** 的指南。

🌐 **语言:**  🇬🇧 **英文** 

🤗 **HuggingFace:** [Jackrong](https://huggingface.co/Jackrong)

<br>

[![Unsloth](https://img.shields.io/badge/Powered%20by-Unsloth-8A2BE2?style=flat-square)](https://github.com/unslothai/unsloth)
[![Google Colab](https://img.shields.io/badge/Environment-Google%20Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/Framework-PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Hugging Face](https://img.shields.io/badge/Model%20Hub-Hugging%20Face-FFD21E?style=flat-square&logo=huggingface&logoColor=black)](https://huggingface.co/)
[![LoRA PEFT](https://img.shields.io/badge/Technique-LoRA%20%2F%20PEFT-007EC6?style=flat-square)](#)
[![Beginner Friendly](https://img.shields.io/badge/Level-Beginner%20Friendly-brightgreen?style=flat-square)](#)

</div>

- > ⚠️ **警告**
>
> Kaggle 的临时磁盘空间实际上限制在约 **92 GB**（即使界面显示 **50 GB**，有时也能利用到约 90 GB）。因此，**加载和训练 Qwen3.5-35B-A3B 极具挑战性且容易失败**。
>
> 为避免不必要的问题，建议主要将 Kaggle 用于**学习和演示目的**。如需实际训练或导出模型，请考虑使用 **Google Colab 或本地机器**。
>
> 虽然部分 Kaggle 账户可能拥有 **H100 GPU** 的访问权限，但存储空间仍然是主要瓶颈。如果您仍选择在 Kaggle 上训练，请考虑以下方法：
>
> - **方案一（推荐）：**  
>   先上传 **16 位模型**，然后使用 **Colab 量化笔记本**生成所有 GGUF 变体。Colab 通常提供更多临时存储空间。
>
> - **方案二（基于检查点的工作流程）：**  
>   - 将训练检查点保存在 `output/` 目录中。  
>   - 将 `output/` 文件夹上传到 **Kaggle 数据集**。  
>   - 在另一台机器（或环境）上，使用 **Kaggle CLI** 下载数据集。  
>   - 加载检查点，合并模型，并在该环境中进行导出/量化操作。
>
> - 此工作流程有助于绕过 Kaggle 磁盘限制，并允许您在具有更多可用存储空间的机器上完成整个过程。🚀
---

## 开始之前：所需 API 密钥 🔑

如果您是 Kaggle 笔记本的新手，请在运行以下单元格之前准备**两个 API 密钥**，并先将它们存储在 **Kaggle Secrets** 中。

### 1. `WANDB_API_KEY`

- 此密钥用于登录 **Weights & Biases (W&B)**。
- 在本笔记本中，它用于实验跟踪、日志记录和训练可视化。
- 如果没有它，开头的 W&B 登录单元格将失败。

### 2. `HF_TOKEN`

- 此密钥用于登录 **Hugging Face**。
- 在本笔记本中，如果您稍后想要**将训练好的模型或 GGUF 文件上传到 Hugging Face Hub**，则主要需要此密钥。
- 如果您只想在 Kaggle 内部进行训练，并且不打算上传工件，则可能不需要立即使用此密钥，但仍建议提前准备好。

### 如何将它们存储在 Kaggle Secrets 中

1. 打开您的 Kaggle 笔记本。
2. 打开右侧的 **Secrets** 面板。
3. 按以下确切名称添加密钥：
   - `WANDB_API_KEY`
   - `HF_TOKEN`
4. 为每个密钥粘贴相应的值。
5. 保存密钥，然后返回并运行笔记本。

### 初学者提示 ✨

- 保持密钥**名称**与代码期望的完全一致。
- 不要将 API 密钥直接粘贴到笔记本代码单元格中。
- 使用 Kaggle Secrets 是更安全、更简洁的管理凭据的方式。

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
import wandb 
wandb.login(key=secret_value_0)
print("Attempted to log in to W&B.")
import os
output_directory = "/kaggle/working/"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
print(f"Checkpoints will be saved to: {output_directory}")

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [ ]:
#!cp -r /kaggle/input/datasets/your-username/your-35b-checkpoint /kaggle/working/

In [ ]:
%%capture
! pip uninstall unsloth unsloth_zoo -y
! pip install git+https://github.com/unslothai/unsloth-zoo.git --no-deps
! pip install git+https://github.com/unslothai/unsloth.git --no-deps

In [ ]:
import os
# Note that to get best performance on A100, we needed to install causal-conv1d
# For this to not take too long, we had to downgrade to torch 2.8 (from torch 2.9)
# This means we wouldn't be able to use torch's grouped_mm here
# so we fallback to unsloth triton kernels for MoE
# We need to disable autotuning to save both time and memory for the colab notebook.
# If you are trying this elsewhere, we might recommend
# you install Flash Attention, Flash Linear Attention and CausalConv1d with torch 2.9
# `!uv pip install --no-build-isolation flash-attn flash-linear-attention causal_conv1d==1.6.`
# You can even try playing around with the below env var for faster performance but make sure you have enough VRAM to try autotuning.
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE'] = '1'

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower, 8, 16, 32, 64, 128, depends on your GPU VRAM

model, processor = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-35B-A3B", # This is a very big model, might take a while for downloading # You can use any model from the list above and HF will download it for you. Depends on your GPU memory.
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False, # Not supported for MoE (yet!)
)
tokenizer = processor.tokenizer # To tokenize text

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # LoRA rank: controls the size of the low-rank adaptation matrices. Higher r = more capacity, but more VRAM usage and slower training. Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "gate_up_proj", #Enable LoRA on MoE layers
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = True, # Reduces memory usage
    random_state = 3407,
    bias = "none",
)

## 数据集处理概览 📚

本节主要旨在演示本Qwen3.5-35B-A3B笔记本中**SFT数据准备的工作流程和核心思路**，而非提供单一的"正确"数据配方。以下的数据集选择、目标样本量和混合策略均属于当前以推理为中心的演示配置。在实际项目中，您应将其替换为更符合自身任务目标、质量标准及计算预算的数据集与配比。

### 本流程的核心步骤

1. **首先确定数据集和目标样本量**
   本笔记本当前使用三个数据集：
   - `ds1` = `nohurry/Opus-4.6-Reasoning-3000x-filtered`：目标 `3900` 条样本
   - `ds2` = `Jackrong/Qwen3.5-reasoning-700x`：目标 `700` 条样本
   - `ds3` = `Roman1111111/claude-opus-4.6-10000x`：目标 `10000` 条样本
   加载器使用 `min(sample_count, len(ds))`，因此若数据集小于配置目标，实际采样数量可能更低。

2. **加载并随机采样原始数据**
   三个数据集均直接通过 `load_dataset()` 加载，然后使用固定随机种子进行打乱和采样。

3. **将不同数据模式统一为 `conversations` 格式**
   - `ds1` 由 `problem / thinking / solution` 字段构建。
   - `ds2` 从 `conversation=[{from, value}, ...]` 转换而来。
   - `ds3` 按 `messages=[{role, content}, ...]` 处理。

   助手回复被标准化为：

   ```text
   <think>...</think>
   final answer
   ```

4. **过滤空或无效的对话**
   格式转换步骤后，在合并前移除缺少或包含空 `conversations` 的行。

5. **合并并打乱组合数据集**
   笔记本通过 `concatenate_datasets()` 合并所有三个处理后的数据集，然后对合并结果进行打乱。

6. **应用对话模板**
   使用 `qwen3-thinking` 模板将每条对话转换为训练时使用的最终 `text` 字段。

7. **按序列长度过滤并验证助手格式**
   移除长度超过 `8192` 个token的样本，并最终验证助手轮次是否仍遵循 `<think>...</think>\n...` 结构。

### 关于当前混合策略 ⚖️

- 配置的目标比例约为 `3900 : 700 : 10000`。
- 实际混合中，`Roman1111111/claude-opus-4.6-10000x` 权重最大，其余两个数据集作为补充来源。
- 此设置旨在演示如何将多个推理型数据源归一化到同一训练流程中，而非声称这是最优生产混合方案。

### 重要说明 ✨

- 重点在于**流程**：如何加载、采样、标准化、模板化、合并、过滤和验证数据集。
- 重点也在于**流程背后的逻辑**：为何需要在训练前统一多个数据集模式。
- 数据集选择与目标样本量均属于当前35B演示配置的一部分。
- 您可以替换数据集、调整样本量或重新平衡混合比例，以更好地匹配自身用例和计算预算。🚀

In [ ]:
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template
import re
import multiprocessing as mp

# ==========================================
# 1. Configuration Area
# ==========================================
RANDOM_SEED = 1234 # Set random seed for reproducibility.
MAX_CONTEXT_WINDOW = 8192  # Same with the model's context window.

num_samples_dict = {
    "ds1": 3900,   # nohurry/Opus-4.6-Reasoning-3000x-filtered
    "ds2": 700,    # Jackrong/Qwen3.5-reasoning-700x
    "ds3": 10000,  # Roman1111111/claude-opus-4.6-10000x
}

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-thinking",
)

# ==========================================
# 2. Load and Sample Datasets
# ==========================================
print("Loading and sampling datasets...")

def load_and_sample(dataset_name, sample_count=None, split="train", subset=None):
    print(f"Processing: {dataset_name} (Target: {sample_count})")
    if subset:
        ds = load_dataset(dataset_name, subset, split=split)
    else:
        ds = load_dataset(dataset_name, split=split)

    if sample_count is not None:
        sample_count = min(sample_count, len(ds))
        ds = ds.shuffle(seed=RANDOM_SEED).select(range(sample_count))

    print(f"{dataset_name} actually sampled: {len(ds)}")
    return ds

ds1 = load_and_sample(
    "nohurry/Opus-4.6-Reasoning-3000x-filtered",
    num_samples_dict["ds1"],
    split="train",
)

ds2 = load_and_sample(
    "Jackrong/Qwen3.5-reasoning-700x",
    num_samples_dict["ds2"],
    split="train",
)

ds3 = load_and_sample(
    "Roman1111111/claude-opus-4.6-10000x",
    num_samples_dict["ds3"],
    split="train",
)

# ==========================================
# 3. Unify conversations, and force assistant = "<think>...</think>\n..."
# ==========================================
def _strip(x):
    return (x or "").strip()

THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL)

def normalize_assistant_to_think_solution(text: str) -> str:
    text = _strip(text)

    if not text:
        return "<think></think>\n"

    m = THINK_BLOCK_RE.search(text)
    if m:
        think_block = m.group(0).strip()
        rest = text[m.end():]
        rest = rest.lstrip()
        return f"{think_block}\n{rest}".rstrip() if rest else f"{think_block}\n"
    else:
        return f"<think></think>\n{text}".rstrip()

def format_ds1(examples):
    problems  = examples.get("problem", [])
    thinkings = examples.get("thinking", [])
    solutions = examples.get("solution", [])

    out = []
    for p, t, s in zip(problems, thinkings, solutions):
        p = _strip(p)
        t = _strip(t)
        s = _strip(s)

        if not p or not s:
            continue

        assistant = f"<think>{t}</think>\n{s}" if t else f"<think></think>\n{s}"

        out.append([
            {"role": "user", "content": p},
            {"role": "assistant", "content": assistant},
        ])

    return {"conversations": out}

def format_ds2(examples):
    convos_list = examples.get("conversation", [])
    out = []

    for conv in convos_list:
        if not conv:
            continue

        cleaned = []
        for m in conv:
            frm = (m.get("from") or "").strip()
            val = m.get("value", "")

            if frm == "human":
                cleaned.append({"role": "user", "content": _strip(val)})
            elif frm == "gpt":
                cleaned.append({"role": "assistant", "content": normalize_assistant_to_think_solution(val)})
            else:
                continue

        if len(cleaned) < 2 or cleaned[-1]["role"] != "assistant":
            continue

        out.append(cleaned)

    return {"conversations": out}

def format_ds3(examples):
    """
    ds3 assumed structure:
    messages=[{role, content}, ...]
    """
    messages_list = examples.get("messages", [])
    out = []

    for msgs in messages_list:
        if not msgs:
            continue

        convo = [m for m in msgs if m.get("role") != "system"]
        if len(convo) < 2 or convo[-1].get("role") != "assistant":
            continue

        cleaned = []
        for m in convo:
            role = m.get("role")
            content = m.get("content", "")
            if role == "assistant":
                content = normalize_assistant_to_think_solution(content)
            else:
                content = _strip(content)
            cleaned.append({"role": role, "content": content})

        out.append(cleaned)

    return {"conversations": out}

print("Normalizing format and enforcing <think>...</think>\\n... structure...")

ds1 = ds1.map(format_ds1, batched=True, remove_columns=ds1.column_names)
ds2 = ds2.map(format_ds2, batched=True, remove_columns=ds2.column_names)
ds3 = ds3.map(format_ds3, batched=True, remove_columns=ds3.column_names)
ds1 = ds1.filter(lambda x: x["conversations"] is not None and len(x["conversations"]) > 0)
ds2 = ds2.filter(lambda x: x["conversations"] is not None and len(x["conversations"]) > 0)
ds3 = ds3.filter(lambda x: x["conversations"] is not None and len(x["conversations"]) > 0)

# ==========================================
# 4. Merge + Shuffle
# ==========================================
print("Merging and shuffling datasets...")
combined_dataset = concatenate_datasets([ds1, ds2, ds3]).shuffle(seed=RANDOM_SEED)
print(f"Total entries after merge: {len(combined_dataset)}")

# ==========================================
# 5. Apply Chat Template (Generating the text column)
# ==========================================
print("Applying Chat Template (qwen3.5)...")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = combined_dataset.map(formatting_prompts_func, batched=True)

# ==========================================
# 6. Filter by Token Length (Multiprocessing)
# ==========================================
num_proc = mp.cpu_count()
print(f"Detected {num_proc} CPU cores")
print(f"Using {num_proc} processes to filter out samples larger than {MAX_CONTEXT_WINDOW} tokens...")

_text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def filter_long_sequences_batched(examples):
    texts = examples["text"]
    tokenized = _text_tok(
        texts,
        truncation=False,
        padding=False,
        add_special_tokens=False,
    )["input_ids"]
    return [len(toks) <= MAX_CONTEXT_WINDOW for toks in tokenized]

before_count = len(dataset)
dataset = dataset.filter(
    filter_long_sequences_batched,
    batched=True,
    num_proc=num_proc
)
after_count = len(dataset)

print("------------------------------------------------")
print(f"Total amount before filtering: {before_count}")
print(f"Total amount after filtering: {after_count}")
print(f"Number of samples removed: {before_count - after_count}")
print("------------------------------------------------")

# ==========================================
# 7. Verification
# ==========================================
def check_assistant_format(examples):
    convos = examples["conversations"]
    ok = []
    for convo in convos:
        good = True
        for m in convo:
            if m["role"] == "assistant":
                c = m.get("content", "")
                if "<think>" not in c or "</think>" not in c:
                    good = False
                    break
                if not re.search(r"</think>\n", c):
                    good = False
                    break
        ok.append(good)
    return {"_ok": ok}

check = dataset.map(
    check_assistant_format,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=num_proc
)
bad = len(check) - sum(check["_ok"])
print(f"Format verification complete: Number of samples mismatching '<think>...</think>\\n...'   {bad}")

if bad > 0:
    dataset = dataset.filter(lambda x: all(
        (m["role"] != "assistant") or (("<think>" in m["content"]) and ("</think>\n" in m["content"]))
        for m in x["conversations"]
    ))

# ==========================================
# 8. Final Preview
# ==========================================
print("\nSample preview (text):")
print("=" * 80)
print(dataset[0]["text"][:800])
print("=" * 80)

print("\nSample preview (last assistant content):")
last_asst = [m for m in dataset[0]["conversations"] if m["role"]=="assistant"][-1]["content"]
print(last_asst[:800])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2, # Number of samples processed on each device (GPU) in one forward/backward pass.
        gradient_accumulation_steps = 4,  # Accumulate gradients over 6 steps before updating weights; simulates a larger batch size without needing more VRAM.
        warmup_ratio = 0.04,  # Use the first 3%-5% of total training steps to gradually increase the learning rate for more stable training.
        #warmup_steps = 60,
        num_train_epochs = 2, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        save_steps = 100,  # Save a checkpoint every 100 training steps.You can adjust this as needed.
        save_total_limit = 1, # Keep only the most recent checkpoint; older ones are deleted to save disk space.
        save_strategy = "steps",
        report_to = "wandb", # Can use Weights & Biases
        output_dir = output_directory,
    ),
)

In [ ]:
dataset[100]['text']

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# Compilation can take 2-3 minutes of time, so please be patient :)
trainer.train()
# If training is interrupted, you can resume with:
# trainer.train(resume_from_checkpoint=True)  # auto-load latest checkpoint or trainer.train(resume_from_checkpoint="checkpoint-xxx")  # specify a checkpoint path

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

In [ ]:
from kaggle_secrets import UserSecretsClient
import json
import os
from huggingface_hub import whoami

# Retrieve the Hugging Face token from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN not found. Add a secret named 'HF_TOKEN' in your Kaggle settings.")
    raise e

print("Successfully retrieved HF_TOKEN from Kaggle Secrets.")

# Resolve the Hugging Face username and target repository ID
try:
    user_info = whoami(token=hf_token)
    username = user_info["name"]
    repo_id = f"{username}/Qwopus-3.5-35B-A3B"
    print(f"Authenticated as: {username}")
    print(f"Target repository: {repo_id}")
except Exception as e:
    print("Authentication failed. Check the token and network connectivity.")
    raise e

# Upload the merged 16-bit model
print("Uploading merged 16-bit model. This may take a few minutes.")

model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token
)

print(f"Upload complete: https://huggingface.co/{repo_id}")


## 导出GGUF模型前的注意事项 ⚠️

- 请注意，Kaggle笔记本可用的临时存储空间实际上限约为**92 GB**，尽管界面可能显示为**50 GB**。
- 由于此限制，直接在Kaggle中完成大型35B GGUF模型的导出可能较为困难。
- 一个实用的替代方案是：先上传**16位模型**，然后使用我发布的**Colab量化笔记本**，在Colab中完成量化并导出全套GGUF规格文件。
- Colab通常提供**更大的临时存储空间**，因此更适合完整的多规格GGUF导出工作流。🚀

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

# Retrieve the Hugging Face token from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN not found. Add a secret named 'HF_TOKEN' in your Kaggle settings.")
    raise e

print("Successfully retrieved HF_TOKEN from Kaggle Secrets.")

# Resolve the Hugging Face username and target repository ID
try:
    user_info = whoami(token=hf_token)
    username = user_info["name"]
    repo_id = f"{username}/Qwopus-3.5-35B-A3B-GGUF"
    print(f"Authenticated as: {username}")
    print(f"Target repository: {repo_id}")
except Exception as e:
    print("Authentication failed. Check the token and network connectivity.")
    raise e

# Convert and upload the GGUF model artifacts
print("Converting and uploading GGUF artifacts (q8_0). This may take some time.")

model.push_to_hub_gguf(
    repo_id,
    tokenizer,
    quantization_method=["q8_0"],
    token=hf_token
)

print(f"GGUF upload complete: https://huggingface.co/{repo_id}")
